# Top‑Down Product Analysis with 4 bar charts. 


- Top Categories by Revenue
- Top Categories by Units Sold
- Top Products by Revenue
- Top Products by Units Sold


In [ ]:
import pandas as pd

# Load datasets
orders = pd.read_csv("../Source data/OListDatasets/olist_orders_dataset.csv")
order_items = pd.read_csv("../Source data/OListDatasets/olist_order_items_dataset.csv")
products = pd.read_csv("../Source data/OListDatasets/olist_products_dataset.csv")

# Inspect columns
print("Order dataset columns:", orders.columns)
print("Order_items dataset columns:", order_items.columns)
print("Products dataset columns:", products.columns)

# selection of language
#category_col = "product_category_name_english" if "product_category_name_english" in products.columns else "product_category_name"
category_col =  "product_category_name"

# Merge order items with products
order_items = order_items.merge(products, on='product_id', how='left')

# Step 1: Category-level revenue
category_sales = (
    order_items.groupby(category_col)['price']
    .sum()
    .reset_index()
    .sort_values('price', ascending=False)
)

print("\nTop Categories by Revenue:")
print(category_sales.head(10))

# Step 2: Product-level revenue
product_sales = (
    order_items.groupby(['product_id', category_col])['price']
    .sum()
    .reset_index()
    .sort_values('price', ascending=False)
)

print("\nTop Products by Revenue:")
print(product_sales.head(10))

# Step 3: Units sold per category
category_units = (
    order_items.groupby(category_col)['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id':'units_sold'})
    .sort_values('units_sold', ascending=False)
)

print("\nTop Categories by Units Sold:")
print(category_units.head(10))

Order dataset columns: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Order_items dataset columns: Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Products dataset columns: Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

Top Categories by Revenue:
     product_category_name       price
11            beleza_saude  1258681.34
66      relogios_presentes  1205005.68
13         cama_mesa_banho  1036988.68
32           esporte_lazer   988048.97
44  informatica_acessorios   911954.32
54        moveis_decoracao   729762.49
26   

In [7]:
import plotly.express as px

# 1. Top Categories by Revenue
fig1 = px.bar(
    category_sales.head(10),
    x=category_col, y='price',
    title="Top 10 Product Categories by Revenue",
    labels={category_col: "Product Category", "price": "Revenue"},
    text='price'
)
fig1.update_traces(texttemplate='%{text}', textposition='outside')  # show full numbers
fig1.update_layout(template="plotly_white", height=500, xaxis_tickangle=-45)
fig1.show()

# 2. Top Categories by Units Sold
fig2 = px.bar(
    category_units.head(10),
    x=category_col, y='units_sold',
    title="Top 10 Product Categories by Units Sold",
    labels={category_col: "Product Category", "units_sold": "Units Sold"},
    text='units_sold'
)
fig2.update_traces(texttemplate='%{text}', textposition='outside')  # show full numbers
fig2.update_layout(template="plotly_white", height=500, xaxis_tickangle=-45)
fig2.show()

# 3. Top Products by Revenue (shortened product_id to 10 chars)
product_sales['product_label'] = (
    product_sales[category_col] + " – " + product_sales['product_id'].str[:10]
)

fig3 = px.bar(
    product_sales.head(10),
    x='product_label', y='price', color=category_col,
    title="Top 10 Products by Revenue",
    labels={"product_label": "Product (Category – ID)", "price": "Revenue"},
    text='price'
)
fig3.update_traces(texttemplate='%{text}', textposition='outside')  # show full numbers
fig3.update_layout(template="plotly_white", height=500, xaxis_tickangle=-45, legend_title_text="Category")
fig3.show()

# 4. Top Products by Units Sold (shortened product_id to 10 chars)
product_units = (
    order_items.groupby(['product_id', category_col])['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id':'units_sold'})
    .sort_values('units_sold', ascending=False)
    .head(10)
)
product_units['product_label'] = (
    product_units[category_col] + " – " + product_units['product_id'].str[:10]
)

fig4 = px.bar(
    product_units,
    x='product_label', y='units_sold', color=category_col,
    title="Top 10 Products by Units Sold",
    labels={"product_label": "Product (Category – ID)", "units_sold": "Units Sold"},
    text='units_sold'
)
fig4.update_traces(texttemplate='%{text}', textposition='outside')  # show full numbers
fig4.update_layout(template="plotly_white", height=500, xaxis_tickangle=-45, legend_title_text="Category")
fig4.show()